In [9]:
%pip install azure-storage-blob

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.28.0 requires packaging<24,>=16.8, but you have packaging 24.2 which is incompatible.
streamlit 1.28.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
  Using cached azure_core-1.39.0-py3-none-any.whl.metadata (48 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)
Using cached azure_core-1.39.0-py3-none-any.whl (218 kB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ------------ --------------------------- 1.0/3.5 MB 7.2 MB/s eta 0:00:01
   --------------------------------- ------ 2.9/3.5 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 8.2 MB/s eta 0:00:00
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

  Attempting uninstall: typing-extensions

    Found existing inst

In [ ]:
import requests
import os
import json
from dotenv import load_dotenv
import pandas as pd
from azure.storage.blob import BlobServiceClient

# Load the variables from the .env file
load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")
AZURE_CONNECTION = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Sua lista de cidades
CITYS = ["Porto Alegre", "Sao Paulo", "Curitiba", "Rio de Janeiro", "Florianopolis", "Guaiba"]

In [ ]:
# Azure Data Lake
try:
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION)
    container_client = blob_service_client.get_container_client("datalake")
    print("Successfully connected to Azure!")
except Exception as e:
    print(f"Error connecting to Azure. Check your connection string in .env. Details: {e}")
    exit()

Get lon/lat

In [2]:
values = []
for city_loc in CITYS:
    url_lon_lat = f"http://api.openweathermap.org/geo/1.0/direct?q={city_loc}&limit={2}&appid={API_KEY}"
    response_cit = requests.get(url_lon_lat)

    values_temp = {"City" : city_loc,
                   "lat" : response_cit.json()[0]['lat'],
                   "lon" : response_cit.json()[0]['lon']}
    values.append(values_temp)

values

[{'City': 'Porto Alegre', 'lat': -30.0324999, 'lon': -51.2303767},
 {'City': 'Sao Paulo', 'lat': -23.5506507, 'lon': -46.6333824},
 {'City': 'Curitiba', 'lat': -25.4295963, 'lon': -49.2712724},
 {'City': 'Rio de Janeiro', 'lat': -22.9110137, 'lon': -43.2093727},
 {'City': 'Florianopolis', 'lat': -27.5973002, 'lon': -48.5496098}]

In [ ]:
base_climat = []

for city in values:
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={city['lat']}&lon={city['lon']}&appid={API_KEY}&units=metric&lang=pt_br"

    # Place the order
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print(f"Sucesso! Dados recebidos: {city['City']}")
        
        # START OF INTEGRATION WITH AZURE (BRONZE LEVEL)
        try:
            # About JSON
            json_data = json.dumps(data, ensure_ascii=False, indent=4)
            
            # Remove spaces from the city name to avoid errors in the URL
            nome_arquivo = city["City"].replace(' ', '_')
            caminho_no_azure = f"bronze/weather_{nome_arquivo}.json"
            
            # Upload to Data Lake Gen2
            blob_client = container_client.get_blob_client(caminho_no_azure)
            blob_client.upload_blob(json_data, overwrite=True)
            print(f"Save to Azure Data Lake: {caminho_no_azure}")
        except Exception as e:
            print(f"Error saving to Azure: {e}")

        # Simulated Silver layer in Pandas
        base_climat.append(pd.DataFrame({"ID": [data["id"]],
                                         "city": [city["City"]],
                                         "lat": [data["coord"]["lat"]],
                                         "lon": [data["coord"]["lon"]],
                                         "weather_main": [data["weather"][0]["main"]],
                                         "weather_description": [data["weather"][0]["description"]],
                                         "weather_icon": [data["weather"][0]["icon"]],
                                         "temperature perception": [data["main"]["feels_like"]],
                                         "minimum temperature": [data["main"]["temp_min"]],
                                         "maximum temperature": [data["main"]["temp_max"]],
                                         "humidity": [data["main"]["humidity"]],
                                         "sea_level": [data.get("main", {}).get("sea_level", "N/A")], # Using .get() to avoid an error if sea_level is not present
                                         "visibility": [data["visibility"]],
                                         "wind_speed": [data["wind"]["speed"]],
                                         "wind_deg": [data["wind"]["deg"]],
                                         "clouds": [data["clouds"]["all"]],
                                         "dataTime": [data["dt"]],
                                         "country": [data["sys"]["country"]],
                                         "sunrise": [data["sys"]["sunrise"]],
                                         "sunset": [data["sys"]["sunset"]]}))
    else:
        print(f"Erro: {response.status_code}")
        print(response.text)

Sucesso! Dados recebidos:  Porto Alegre
Sucesso! Dados recebidos:  Sao Paulo
Sucesso! Dados recebidos:  Curitiba
Sucesso! Dados recebidos:  Rio de Janeiro
Sucesso! Dados recebidos:  Florianopolis


In [4]:
df_dados_agrupados = pd.concat(base_climat)

In [5]:
df_dados_agrupados

,ID,city,lat,lon,weather_main,weather_description,weather_icon,temperature perception,minimum temperature,maximum temperature,humidity,sea_level,visibility,wind_speed,wind_deg,clouds,dataTime,country,sunrise,sunset
0,3452925,Porto Alegre,-30.0325,-51.2304,Clear,céu limpo,01d,40.83,34.78,36.26,47,1004,10000,3.60,80,0,1774118330,BR,1774085331,1774128919
0,3458611,Sao Paulo,-23.5507,-46.6334,Clouds,nuvens dispersas,03d,30.57,27.79,28.69,64,1011,10000,5.66,160,40,1774118211,BR,1774084231,1774127813
0,3464975,Curitiba,-25.4296,-49.2713,Clouds,nuvens dispersas,03d,30.77,28.96,31.29,53,1010,10000,3.60,210,40,1774117804,BR,1774084864,1774128446
0,3451190,Rio de Janeiro,-22.9110,-43.2094,Clouds,algumas nuvens,02d,34.53,27.99,29.73,79,1011,10000,7.20,190,20,1774118401,BR,1774083410,1774126991
0,3463237,Florianopolis,-27.5973,-48.5496,Clouds,algumas nuvens,02d,31.28,27.84,29.96,68,1008,10000,5.66,350,20,1774118306,BR,1774084689,1774128274
